In [10]:
#import modules
import pyodbc
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import pandas as pd

#server credentials - prod
prod_server = 'caisrv38'
prod_db = 'WH_BW'

#server credentials - docker
docker_server = 'localhost'
docker_db = 'test_db'
username = 'sa' 
password = '-xyKTjBx3Gk1k9zi5I39!'
#-------------------------
driver = 'SQL Server' 
schema = 'dbo'


#sql connection - prod 
prod_cnxn = pyodbc.connect(
    Trusted_Connection= 'Yes',
    Driver= f'{driver}',
    Server= prod_server,
    database= prod_db,

)
cursor = prod_cnxn.cursor()



In [31]:
#docker connection
docker_cnxn = pyodbc.connect(
    # Trusted_Connection= 'Yes',
    Driver= '{SQL Server}',
    Server= docker_server,
    UID = username,
    PWD = password
    # database= db
)

docker_connection = f"DRIVER={driver};SERVER={docker_server};DATABASE=test_db;UID={username};PWD={password}"
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": docker_connection})

engine = create_engine(connection_url, fast_executemany = True)

docker_cnxn.autocommit = True
# test conditions
# docker_cnxn.execute("""alter database test_db set single_user with rollback immediate;""")
# docker_cnxn.execute('drop database test_db;')
# -------------------------------------------
docker_cnxn.execute('''
if not exists (select 1 from sys.databases where name = N'test_db')
create database test_db;
''')
docker_cnxn.autocommit = False

#create a list of each table in the database, and remove table names from the list that contain numbers (i.e duplicates/backups with dates on the end)
WH_BW_tables = [table.table_name for table in cursor.tables()]
WH_BW_tables = [item for item in WH_BW_tables if not any(char.isdigit() for char in item)]
# print(WH_BW_tables[50:60])

In [32]:
# for table in WH_BW_tables[:50]:
#     try:
#         #read
#         query = f'select top 1000 * from {prod_db}.{schema}.{table}'
#         results = cursor.execute(query).fetchall()
#         df_sql = pd.read_sql(query, prod_cnxn)

#         #write
#         df_sql.to_sql(f'{table}', schema='dbo', 
#                     con = engine, chunksize=1000, 
#                     index=False, if_exists='replace')
#     except Exception:
#         print(f'failed to insert {table} to docker container')

table = 'wms_contcase_tbl'
query = f'select top 1000 * from {prod_db}.{schema}.{table}'
results = cursor.execute(query).fetchall()
df_sql = pd.read_sql(query, prod_cnxn)

df_sql.head()

,gl_cmp_key,in_whs_key,wms_conthdr_key,wms_contdtl_key,wms_contcase_key,wms_contcase_ctwgt,wms_contcase_acwgt,in_sublot_key,wms_contcase_prddt,created_on,...,pieces,lineKey,userKey,wms_contcase_orgwhs,wms_contcase_uom,wms_contcase_tbl_id,dbserverDateTime,wms_contcase_tare,wms_contcase_shipstatus,receiptType
0,CASL,DFDS,*117*NEW**,1,1009582 ...,24.56,25.11,1009582,2021-03-22 14:32:51,2021-03-23 15:13:49,...,9,902,None,YPS,KG,340594868,2021-03-23 15:13:49.6633333,0.0,ACTIVE ...,...
1,CASL,DFDS,*117*NEW**,1,1175959 ...,23.53,24.08,1175959,2021-03-22 14:16:37,2021-03-23 15:13:47,...,9,902,None,YPS,KG,340594865,2021-03-23 15:13:46.9133333,0.0,ACTIVE ...,...
2,CASL,DFDS,*117*NEW**,2,1510341 ...,21.75,22.27,1510341,2021-03-22 14:34:21,2021-03-23 15:18:52,...,5,901,None,OPS,KG,340595251,2021-03-23 15:18:52.0000000,0.0,ACTIVE ...,...
3,CASL,DFDS,*117*NEW**,2,1510342 ...,23.08,23.60,1510342,2021-03-22 14:34:33,2021-03-23 15:18:37,...,5,901,None,OPS,KG,340595236,2021-03-23 15:18:37.0766667,0.0,ACTIVE ...,...
4,CASL,DFDS,*117*NEW**,2,1510349 ...,23.48,24.00,1510349,2021-03-22 14:36:26,2021-03-23 15:18:46,...,5,901,None,OPS,KG,340595246,2021-03-23 15:18:46.7966667,0.0,ACTIVE ...,...


In [35]:

df_sql.to_sql(f'{table}', schema= f'{schema}', 
            con = engine, chunksize=1, 
            index=False, if_exists='replace')

In [3]:
#docker connection
docker_cnxn = pyodbc.connect(
    # Trusted_Connection= 'Yes',
    Driver= '{SQL Server}',
    Server= docker_server,
    UID = username,
    PWD = password
    # database= db
)

In [11]:
docker_cnxn.execute("""alter database test_db set single_user with rollback immediate;""")
docker_cnxn.execute('drop database test_db;')

OperationalError: ('08S01', '[08S01] [Microsoft][ODBC SQL Server Driver]Communication link failure (0) (SQLExecDirectW)')

In [8]:
drivers = [item for item in pyodbc.drivers()]
# driver = drivers[-1]
print("driver:{}".format(drivers))
# server = 'myserver'
# database = 'mydb'
# uid = 'myuser'
# pwd = 'mypass'
# con_string = f'DRIVER={driver};SERVER={server};DATABASE={database};UID={uid};PWD={pwd}'
# print(con_string)
# cnxn = pyodbc.connect(con_string)

driver:['SQL Server', 'SQL Server Native Client 11.0', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client RDA 11.0', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


In [58]:
# for table in cursor.tables(table = 'wms_contcase_tbl'):
    # table = table.table_name
    # print(table)
# query = f'select top 1000 * from {db}.{schema}.{table}'
# df_sql = pd.read_sql(query, cnxn)
# print(df_sql.head())
# print(df_sql.columns)
# df_sql.to_csv('wms_contcase_tbl_dev.csv')
# df_sql.to_sql(f'{table}', engine)

In [37]:
# server = 'localhost' 
# db = 'test_db' 
# username = 'sa' 
# password = '4bzLUKMVV!A3txEk69?*@' 
# cnxn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+db+';UID='+username+';PWD='+ password)
# cursor = cnxn.cursor()

# cursor.execute(f'create table {table}')

db = 'WH_BW'
schema = 'dbo'
table = 'wms_contcase_tbl'
query = f'select top 1000 * from {db}.{schema}.{table}'

#load query to dataframe
df_sql = pd.read_sql(query, cnxn)
df_sql.head()

query = f'select top 1000 * from {db}.{schema}.{table}'
df_sql = pd.read_sql(query, cnxn)
df_sql.columns



C:\Users\martin.palkovic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pandas\io\sql.py:758: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(
C:\Users\martin.palkovic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\pandas\io\sql.py:758: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


Index(['gl_cmp_key', 'in_whs_key', 'wms_conthdr_key', 'wms_contdtl_key',
       'wms_contcase_key', 'wms_contcase_ctwgt', 'wms_contcase_acwgt',
       'in_sublot_key', 'wms_contcase_prddt', 'created_on', 'changed_on',
       'created_by', 'changed_by', 'lock_count', 'wms_contcase_stat',
       'wms_contcase_lbldt', 'lot', 'allocation', 'pieces', 'lineKey',
       'userKey', 'wms_contcase_orgwhs', 'wms_contcase_uom',
       'wms_contcase_tbl_id', 'dbserverDateTime', 'wms_contcase_tare',
       'wms_contcase_shipstatus', 'receiptType'],
      dtype='object')

In [22]:
#import modules
import pyodbc, urllib
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import pandas as pd

#server credentials
server = 'localhost'
db = 'WH_BW'
username = 'sa' 
password = '4bzLUKMVV!A3txEk69?*@' 


#sql connection - prod 
# prod_cnxn = pyodbc.connect(
#     Trusted_Connection= 'Yes',
#     Driver= '{SQL Server}',
#     Server= server,
#     database= db
# )
# cursor = prod_cnxn.cursor()
# docker_cnxn.autocommit = True

# docker_connection = "DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;UID=sa;PWD=4bzLUKMVV!A3txEk69?*@"
# connection_url = URL.create("mssql+pyodbc", 
#                             query={"odbc_connect": docker_connection})
docker_cnxn = pyodbc.connect(
    # Trusted_Connection= 'Yes',
    Driver= '{SQL Server}',
    Server= server,
    UID = username,
    PWD = password
    # database= db
)

# engine = create_engine(connection_url, fast_executemany = True, connect_args = {'autocommit': True})
# docker_cnxn = engine.connect()
docker_cnxn.autocommit = True
# docker_cnxn.execute('commit')
docker_cnxn.execute('create database test_db')

C:\Users\martin.palkovic\AppData\Local\Temp\ipykernel_4296\753783417.py:27: DeprecationWarning: PyUnicode_FromUnicode(NULL, size) is deprecated; use PyUnicode_New() instead
  docker_cnxn = pyodbc.connect(


In [1]:
driver = 'ODBC Driver 17 for SQL Server'

print(f'{driver}')

ODBC Driver 17 for SQL Server
